# Explore
Explore what subreddits relate to or mention resumes.

In [2]:
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()

True

In [4]:
client_id = os.environ.get("REDDIT_CLIENT_ID")
client_secret = os.environ.get("REDDIT_CLIENT_SECRET")

In [5]:
import praw

client= praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent="research/0.1"
)

## Exploring Subreddits
Found that r/resumes is a prominent subreddit

In [6]:
resume_subs = client.subreddits.search_by_name("resume")

for sub in resume_subs:
    print(f"r/{sub.display_name} - {sub.subscribers:,} subscribers")

r/Resume - 130,812 subscribers
r/resumes - 1,328,408 subscribers
r/ResumeExperts - 29,598 subscribers
r/ResumeTips - 18,118 subscribers
r/ResumeCoverLetterTips - 27,848 subscribers
r/resumesindia - 7,321 subscribers
r/resumesupport - 8,767 subscribers
r/ResumeFairies - 7,082 subscribers
r/resumeph - 16 subscribers
r/resumereview - 2,295 subscribers


## Downloading sample of r/resumes Posts
Retrieved all posts with >=1 comment over time period spanning January 1st, 2026 to November 1st, 2026.

In [13]:
import deepsift

In [38]:
from deepsift.scraper import RedditScraper
from deepsift.filters import Filters
from deepsift.models import PostField

filters = Filters(
    post_include_fields=[
        PostField.ID,
        PostField.TITLE,
        PostField.SELFTEXT,
        PostField.SCORE,
        PostField.NUM_COMMENTS,
        PostField.CREATED_UTC,
        PostField.SUBREDDIT,
        PostField.URL
    ],
    post_filter=lambda post: post.num_comments >= 1,
)

scraper = RedditScraper(
    filters=filters,
    post_workers=4,
    shard_days=30,
)

# uncomment to run extraction
# post_ids = scraper.scrape_posts(
#     subreddits=["resumes"],
#     from_dt="2026-01-01",
#     to_dt="2026-11-01",
#     output_path="data/subreddit_resumes_posts.parquet",
# )

# print(f"Scraped {len(post_ids)} curated posts.")

## Finding Relevant Posts in Retrieved Data

In [285]:
import pandas as pd

posts_path = "data/subreddit_resumes_posts.parquet"
posts = pd.read_parquet(posts_path)

In [286]:
posts.shape

(20827, 8)

In [287]:
# validating the time period was respected
posts["created_utc"] = pd.to_datetime(posts["created_utc"], unit="s")

In [288]:
# this validates that it we did get data for the complete time period requested
posts["created_utc"].describe()

count                         20827
mean     2026-05-14 14:14:26.002256
min             2026-01-01 00:29:21
25%      2026-02-25 15:13:13.500000
50%             2026-05-04 21:39:57
75%             2026-07-28 12:10:49
max             2026-10-31 22:55:34
Name: created_utc, dtype: object

In [290]:
# remove posts with empty selftext, as they are likely to be irrelevant posts
is_nonempty_text = posts["selftext"].str.strip().str.len() > 1
not_empty_posts = posts[is_nonempty_text]
not_empty_posts["selftext"].str.len().describe()

count    18801.000000
mean       345.065156
std        548.040952
min          4.000000
25%          9.000000
50%         54.000000
75%        519.000000
max      11280.000000
Name: selftext, dtype: float64

In [297]:
# checks posts with text less than 25% of the full dataset, as they may be irrelevant posts (found '[removed]' posts and '[deleted]' posts)
is_sufficient_length = not_empty_posts["selftext"].str.len() < not_empty_posts["selftext"].str.len().quantile(0.5)
is_valid_text = posts["selftext"].str.lower().isin(["[removed]", "[deleted]"]) == False
pd.DataFrame(not_empty_posts[is_sufficient_length]["selftext"].value_counts()).head(5)

,count
selftext,
[removed],9071
[deleted],79
Any feedback is appreciated!,3
Roast my resume,3
Looking for feedback on my resume,2


In [295]:
# filter out the rows to only valid posts with sufficient text length
sufficient_text_posts = posts[is_valid_text & is_nonempty_text]
sufficient_text_posts.shape

(9651, 8)

In [299]:
# filter out the duplicates
no_dups_sufficient_text_posts = sufficient_text_posts.drop_duplicates(subset=["selftext"])
no_dups_sufficient_text_posts.shape

(9501, 8)

In [306]:
# check url file types to see if there are any image attachments
endings = pd.DataFrame(no_dups_sufficient_text_posts["url"].str.split(".").str[-1].str.lower().value_counts())
likely_extensions = endings[endings.index.str.len() < 10]
likely_extensions.head(5)

,count
url,
jpeg,2108
png,1676
0,1
cv,1
pro/,1


In [308]:
# only keep posts with image attachments
has_attachment = no_dups_sufficient_text_posts["url"].str.split(".").str[-1].str.lower().isin(["jpeg", "png"])
valid_posts_w_attachments = no_dups_sufficient_text_posts[has_attachment]
valid_posts_w_attachments.shape

(3784, 8)

In [323]:
# only keep posts with a resume type in the title, which is usually in the format of [resume type]
pattern = r"\[(.*)\]"
valid_posts_w_attachments["resume_type"] = valid_posts_w_attachments["title"].str.extract(pattern)
valid_posts_w_attachments.shape

(3708, 9)

In [355]:
# actually parse the resume type from the title using regex
import re

def extract_number(phrase) -> int | None:
    # if number exists in the phrase it returns it else None

    number_exists_pattern = ".*([0-9]+).*"
    number = re.search(number_exists_pattern, phrase)
    print(phrase)
    if number:
        return int(number.group(1))

    return None

def parse_exp_years(phrase) -> int | None:

    if "yoe" in phrase.lower() and len(phrase.strip().split()) == 2:
        print(phrase)
        return extract_number(phrase)
    elif "years" in phrase.lower() and len(phrase.strip().split()) == 2:
        return extract_number(phrase)

    return None



def parse_resume_type(row):

    pattern = r"\[(.*)\]"
    match = re.search(pattern, row["title"])

    if match:
        statement = match.group(1)
        parts = statement.split(",")

        # get years of experience
        years_exp = parse_exp_years(parts[0])
        if years_exp is not None:
            return {"parsed": True, "years_exp": years_exp}
        else:
            return {"parsed": False, "years_exp": parts[0]}
    
    return {}

results = valid_posts_w_attachments.apply(parse_resume_type, axis=1, result_type="expand")
results[results["parsed"] == False]

4 YoE
4 YoE
3 YoE
3 YoE
5 YoE
5 YoE
3 YoE
3 YoE
11 YoE
11 YoE
0 years
7 YoE
7 YoE
3 YoE
3 YoE
2 YoE
2 YoE
7 YoE
7 YoE
0 YoE
0 YoE
3 YoE
3 YoE
5 YoE
5 YoE
5 Years
1.5 Years
4 YoE
4 YoE
0 YoE
0 YoE
0 years
3 YoE
3 YoE
0 YOE
0 YOE
4 YoE
4 YoE
5 YoE
5 YoE
.5 YoE
.5 YoE
0 YoE
0 YoE
0 YoE
0 YoE
4 YoE
4 YoE
4 YoE
4 YoE
0 YoE
0 YoE
0 YoE
0 YoE
2 YoE
2 YoE
1 YoE
1 YoE
1 YoE
1 YoE
1 YoE
1 YoE
5 YoE
5 YoE
2 YoE
2 YoE
0 YoE
0 YoE
2 YoE
2 YoE
0 YOE
0 YOE
7 YOE
7 YOE
0 YoE
0 YoE
1 YoE
1 YoE
0 years
1 YOE
1 YOE
0 YOE
0 YOE
0 YoE
0 YoE
0 YoE
0 YoE
0 years
0 YoE
0 YoE
0 YoE
0 YoE
3 YoE
3 YoE
2 YOE
2 YOE
5 YoE
5 YoE
0 YoE
0 YoE
4 YoE
4 YoE
5 YoE
5 YoE
0 YoE
0 YoE
4 YoE
4 YoE
 YoE 5
 YoE 5
10 YoE
10 YoE
8 YoE
8 YoE
0 YoE
0 YoE
3 YoE
3 YoE
0 YoE
0 YoE
 3 YOE
 3 YOE
1 YoE
1 YoE
1 YoE
1 YoE
10 Years
2+ YoE
2+ YoE
5 YOE
5 YOE
2 YoE
2 YoE
0 years
1 YoE
1 YoE
1 YoE
1 YoE
0 YoE
0 YoE
5 YoE
5 YoE
6 YOE
6 YOE
0 YoE
0 YoE
1 YoE
1 YoE
2 YoE
2 YoE
0 YoE
0 YoE
6 YoE
6 YoE
0 YoE
0 YoE
0 YoE
0 YoE
0 YOE
0 YOE
0 YoE
0 Y

,parsed,years_exp
33,False,13YoE
92,False,4 Months (co-op)
112,False,0Years
141,False,5YoE
156,False,4 YEO
...,...,...
20582,False,0YoE
20654,False,0YE
20687,False,3YOE
20695,False,7YoE


In [344]:
# noticed a common regex pattern in titles of posts
import re

sample_title = clean_posts["title"].iloc[0]
res = re.search(r"\[(.*)\]", sample_title)
res.group(1)

clean_posts["resume_type"] = clean_posts["title"].str.extract(r"\[(.*)\]")
clean_posts.dropna(inplace=True)

/var/folders/my/90dxhc_500vbg9nf8jbt4lq80000gn/T/ipykernel_1940/1603955904.py:9: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  clean_posts.dropna(inplace=True)


In [214]:
clean_posts["resume_type"].value_counts(dropna=False)

resume_type
2 YoE, Unemployed, Digital Marketing, USA                                          7
0 YoE, Unemployed, Retail Assistant, United Kingdom                                5
0 YoE, Unemployed, Artificial Intelligence Engineer, India                         4
0 YoE, Unemployed, Retail Stocker, United States                                   4
0 YoE, Unemployed, Software Engineer, India                                        4
                                                                                  ..
1st Year University Student, intern, Business Field, United State                  1
0 YOE, Data Analyst Co-op, Data Science Co-op, Canada                              1
3 years, intern, intern, USA                                                       1
7 YOE, Employed, System Administrator/Cloud Administrator, USA                     1
1 YoE, IT Business Analyst Intern, Associate/Jr Business Analyst, United States    1
Name: count, Length: 3579, dtype: int64

In [215]:
# 97% of the clean posts have this somewhere in the title, so we can use this to categorize the posts
clean_posts["resume_type"].notna().sum() / clean_posts.shape[0]

np.float64(1.0)

In [211]:
# breaking up the resume type into more granular categories
sample_resume_type = clean_posts["resume_type"].iloc[20]
sample_resume_type.split(",")


['3 YoE', ' Graduate Student', ' MaCC', ' United States']

In [163]:
clean_posts["resume_type"].iloc[20]

'3 YoE, Graduate Student, MaCC, United States'

In [183]:
split_cols = clean_posts["resume_type"].str.split(",", expand=True)

In [167]:
split_cols.info()

<class 'pandas.DataFrame'>
Index: 3712 entries, 3 to 20826
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       3712 non-null   str  
 1   1       3712 non-null   str  
 2   2       3712 non-null   str  
 3   3       3707 non-null   str  
 4   4       175 non-null    str  
 5   5       30 non-null     str  
 6   6       5 non-null      str  
 7   7       2 non-null      str  
dtypes: str(8)
memory usage: 459.5 KB


In [187]:
split_cols[pd.isna(split_cols[7]) == False]

,0,1,2,3,4,5,6,7
670,1 YoE,Recent Graduate,Entry Level Supply Chain Management (Operations,Logistics,Data Analyst,Procurement,Etc.),USA
16813,10 YoE,unemployed,growth,demand,lifecycle,direct,manager/director marketing role,USA


In [198]:
split_cols[pd.isna(split_cols[3]) == False][0].value_counts().head(80)

0
0 YoE      859
1 YoE      275
2 YoE      238
3 YoE      227
0 years    173
          ... 
6YoE         4
5 Yoe        4
3 Years      4
4 Years      4
10+ YoE      4
Name: count, Length: 80, dtype: int64